In [1]:
import sys

sys.path.append("..")


from src.plotting.feature_plotting import plot_graph
import matplotlib.pyplot as plt
import numpy as np
import torch

DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR = "../models"
PLOTS_DIR = "../plots"
from src.data_preparation import load_numpy_files

signal_prefix = f"{DATA_DIR}/sig"
background_prefix = f"{DATA_DIR}/bg"
signal_only_prefix = f"{DATA_DIR}/sig_only"


In [2]:
import src.torch.pre_processing.graph_batching as graph_batching
from importlib import reload
reload(graph_batching)

<module 'src.torch.pre_processing.graph_batching' from '/Users/simi/mu3e_trigger/notebooks_supervised/../src/torch/pre_processing/graph_batching.py'>

In [3]:
hetero_graph_dataset = graph_batching.create_hetero_dataset(
    signal_prefix, has_layer_feature=True
)

In [4]:
from torch_geometric.loader import DataLoader
hetero_graph_data_loader = DataLoader(hetero_graph_dataset, batch_size=512, shuffle=True)

In [5]:
from torch_geometric.data import HeteroData
class EdgeClassifier(torch.nn.Module):
    def __init__(self, node_dims, edge_types, num_layers, hidden_dim = 32):
        super(EdgeClassifier, self).__init__()
        from torch_geometric.nn import HeteroConv, SAGEConv, Linear
        self.convs = torch.nn.ModuleList()
        for i in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv((-1, -1), hidden_dim)
                for edge_type in edge_types
            }, aggr='mean')
            self.convs.append(conv)
        self.lin = Linear(2*hidden_dim, 1)

    def forward(self, hetero_graph: HeteroData):
        x_dict = hetero_graph.x_dict
        edge_index_dict = hetero_graph.edge_index_dict

        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {key: x.relu() for key, x in x_dict.items()}

        # Assuming we want to classify edges of a specific type, e.g., ('pixel', 'to', 'mppc')
        edge_type_to_classify = list(edge_index_dict.keys())[0]  # Just an example
        edge_index = edge_index_dict[edge_type_to_classify]
        src_node_type, _, dst_node_type = edge_type_to_classify

        src_x = x_dict[src_node_type]
        dst_x = x_dict[dst_node_type]

        edge_features = torch.cat([src_x[edge_index[0]], dst_x[edge_index[1]]], dim=-1)
        out = self.lin(edge_features).squeeze()

        return out


In [6]:
edge_classifier = EdgeClassifier(
    node_dims=hetero_graph_dataset.get_node_dims(),
    edge_types=hetero_graph_dataset.get_edge_types(),
    num_layers=5,
    hidden_dim=32
)

In [8]:
from src.torch.training import FocalLoss, HeteroLossWrapper
from sklearn.metrics import roc_auc_score
criterion = HeteroLossWrapper(FocalLoss())
optimizer = torch.optim.Adam(edge_classifier.parameters(), lr=0.001)

from tqdm import tqdm
num_epochs = 10
for epoch in range(num_epochs):
    edge_classifier.train()
    total_loss = 0
    predictions = []
    true_labels = []
    for hetero_graph in tqdm(hetero_graph_data_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        optimizer.zero_grad()
        out = edge_classifier(hetero_graph[0])
        # Assuming labels are stored in hetero_graph.y
        loss = criterion(out, hetero_graph[0]['pixel', 'to', 'mppc'].edge_labels.float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(hetero_graph_data_loader)
    auc = roc_auc_score(true_labels, predictions)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1/10:   3%|▎         | 10/346 [01:12<40:23,  7.21s/it]


IndexError: list index out of range